In [1]:
import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import copy
import random
import matplotlib.pyplot as plt
import unicodedata
from collections import Counter



from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
import Levenshtein as Lev

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

PAD_ID = 0
SOS_ID = 1
EOS_ID = 2





cuda


In [2]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

In [3]:
videos = os.path.join("..", "videos_all_languages.json")

with open(videos, "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.json_normalize(data["videos"])

print(df.columns)
df.head()

Index(['id', 'language', 'Sign', 'filepath', 'HamNoSys', 'concept_url',
       'HandLandmark', 'annotated_video'],
      dtype='object')


,id,language,Sign,filepath,HamNoSys,concept_url,HandLandmark,annotated_video
0,BSL_1,BSL,April,/ephemeral/Project/Data/External/BSL/April.mp4,,https://www.sign-lang.uni-hamburg.de/dicta-sig...,/ephemeral/Project/Data/Interim/BSL/April.json,/ephemeral/Project/Data/Interim/BSL/annotated/...
1,BSL_2,BSL,Athens,/ephemeral/Project/Data/External/BSL/Athens.mp4,,https://www.sign-lang.uni-hamburg.de/dicta-sig...,/ephemeral/Project/Data/Interim/BSL/Athens.json,/ephemeral/Project/Data/Interim/BSL/annotated/...
2,BSL_3,BSL,August,/ephemeral/Project/Data/External/BSL/August.mp4,,https://www.sign-lang.uni-hamburg.de/dicta-sig...,/ephemeral/Project/Data/Interim/BSL/August.json,/ephemeral/Project/Data/Interim/BSL/annotated/...
3,BSL_4,BSL,Berlin,/ephemeral/Project/Data/External/BSL/Berlin.mp4,,https://www.sign-lang.uni-hamburg.de/dicta-sig...,/ephemeral/Project/Data/Interim/BSL/Berlin.json,/ephemeral/Project/Data/Interim/BSL/annotated/...
4,BSL_5,BSL,February,/ephemeral/Project/Data/External/BSL/February.mp4,,https://www.sign-lang.uni-hamburg.de/dicta-sig...,/ephemeral/Project/Data/Interim/BSL/February.json,/ephemeral/Project/Data/Interim/BSL/annotated/...


In [4]:
# =========================
# CELL 3 — Get Landmark Paths
# =========================
Hand_Landmarks_paths = df["HandLandmark"]
Hand_Landmarks = [p for p in Hand_Landmarks_paths if os.path.exists(p)]

print("Valid landmark files:", len(Hand_Landmarks))

Valid landmark files: 3037


In [5]:
# =========================
# CELL 4 — Utility Functions
# =========================
def fix_mojibake(name):
    try:
        name = name.encode("latin1").decode("utf8")
    except:
        pass

    replacements = {
        "ÃŸ": "ß", "Ã„": "Ä", "Ã–": "Ö", "Ãœ": "Ü",
        "Ã¤": "ä", "Ã¶": "ö", "Ã¼": "ü",
        "Ã©": "é", "Ã¨": "è", "Ãª": "ê", "Ã«": "ë",
        "Ã´": "ô", "Ã¢": "â", "Ã¹": "ù", "Ã»": "û",
        "Ã§": "ç",
    }

    for bad, good in replacements.items():
        name = name.replace(bad, good)

    return unicodedata.normalize("NFC", name)


def normalize(seq):
    valid = np.any(seq != 0, axis=1)

    if valid.sum() < 2:
        return np.zeros_like(seq)

    center = seq[valid].mean(axis=0)
    seq[valid] -= center

    scale = np.std(seq[valid])
    if scale < 1e-6:
        return np.zeros_like(seq)

    seq[valid] /= scale
    return seq

In [6]:
HAND_LANDMARK_IDS = list(range(21))
POSE_LANDMARK_IDS = list(range(33))

REDUCED_FACE_LANDMARKS = (
    list(range(70, 103)) +
    list(range(0, 17)) +
    list(range(61, 88)) +
    list(range(152, 172))
)

In [7]:
def convert_frames_to_sequence(frames):
    rows = []

    for frame in frames:
        frame_features = []

        left = {
            lm["id"]: lm
            for hand in frame["hands"] if hand["handedness"] == "Left"
            for lm in hand["landmarks"]
        }

        right = {
            lm["id"]: lm
            for hand in frame["hands"] if hand["handedness"] == "Right"
            for lm in hand["landmarks"]
        }

        for i in HAND_LANDMARK_IDS:
            lm = left.get(i)
            frame_features.extend([0,0,0] if lm is None else [lm["x"], lm["y"], lm["z"]])

        for i in HAND_LANDMARK_IDS:
            lm = right.get(i)
            frame_features.extend([0,0,0] if lm is None else [lm["x"], lm["y"], lm["z"]])

        pose_dict = {lm["id"]: lm for p in frame["pose"] for lm in p["landmarks"]}
        for i in POSE_LANDMARK_IDS:
            lm = pose_dict.get(i)
            frame_features.extend([0,0,0] if lm is None else [lm["x"], lm["y"], lm["z"]])

        face_dict = {lm["id"]: lm for f in frame["face"] for lm in f["landmarks"]}
        for i in REDUCED_FACE_LANDMARKS:
            lm = face_dict.get(i)
            frame_features.extend([0,0,0] if lm is None else [lm["x"], lm["y"], lm["z"]])

        rows.append(np.array(frame_features, dtype=np.float32))

    return np.stack(rows)

In [8]:
# =========================
# CELL 8 — Build Label Dictionary
# =========================
df["filepath"] = df["filepath"].apply(
    lambda p: fix_mojibake(
        os.path.basename(p)
        .replace(".mp4", "")
        .replace(".MOV", "")
    )
)

label_dict = dict(zip(df["filepath"], df["HamNoSys"]))
print("Labels:", len(label_dict))

Labels: 2997


In [9]:
# =========================
# CELL 9 — Load X / y
# =========================
X = []
y = []

for seq_file in os.listdir("processed_sequences"):
    vid = seq_file.replace(".npy", "")
    clean_vid = fix_mojibake(vid)

    if clean_vid in label_dict:
        X.append(np.load(f"processed_sequences/{seq_file}"))
        y.append(label_dict[clean_vid])

print("Samples:", len(X))

Samples: 3055


In [10]:
# =========================
# CELL 7 — Convert JSON Files to .npy
# =========================
os.makedirs("processed_sequences", exist_ok=True)

X, y = [], []

for path in Hand_Landmarks:
    with open(path, "r") as f:
        data = json.load(f)

    vid = fix_mojibake(os.path.basename(data["video_path"]).replace(".mp4",""))

    seq = convert_frames_to_sequence(data["frames"])
    seq = normalize(seq)

    if vid in label_dict:
        X.append(seq)
        y.append(label_dict[vid])

In [11]:
# =========================
# TOKENISATION (FIXED)
# =========================

def tokenize_hamnosys(text):
    """
    Correct character-level tokenisation for HamNoSys.
    Keeps Unicode symbols as individual tokens.
    """
    return list(text.strip())


# build vocabulary
all_tokens = sorted({
    tok
    for seq in y
    for tok in tokenize_hamnosys(seq)
})

token_to_id = {t: i + 3 for i, t in enumerate(all_tokens)}
token_to_id["<PAD>"] = 0
token_to_id["<SOS>"] = 1
token_to_id["<EOS>"] = 2

id_to_token = {v: k for k, v in token_to_id.items()}


vocab_size = len(token_to_id)


# encode sequences
y_encoded = []

for seq in y:
    tokens = tokenize_hamnosys(seq)

    encoded = (
        [SOS_ID] +
        [token_to_id[t] for t in tokens] +
        [SOS_ID]
    )

    y_encoded.append(encoded)

In [12]:
def pad_sequences_X(seqs, max_len):
    feat = seqs[0].shape[1]
    X = torch.zeros(len(seqs), max_len, feat)
    mask = torch.zeros(len(seqs), max_len, dtype=torch.bool)

    for i, s in enumerate(seqs):
        l = min(len(s), max_len)
        X[i, :l] = torch.tensor(s[:l])
        mask[i, :l] = 1

    return X, mask


def pad_sequences_y(seqs, max_len):
    Y = torch.full((len(seqs), max_len), PAD_ID)

    for i, s in enumerate(seqs):
        l = min(len(s), max_len)
        Y[i, :l] = torch.tensor(s[:l])

    return Y

In [13]:
X_train, X_tmp, y_train, y_tmp = train_test_split(X, y_encoded, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.5, random_state=42)

In [14]:

print(np.__version__)
print(torch.__version__)
print(torch.from_numpy(np.array([1,2,3])))

1.26.4
2.5.1+cu121
tensor([1, 2, 3])


In [15]:
# =========================
# CELL 14 — Create Datasets / Loaders (NO *_tensor VARIABLES)
# =========================

max_len_X = max(len(s) for s in X_train + X_val + X_test)
max_len_y = max(len(s) for s in y_train + y_val + y_test)

# create padded tensors directly
train_X, train_mask = pad_sequences_X(X_train, max_len_X)
val_X, val_mask     = pad_sequences_X(X_val, max_len_X)
test_X, test_mask   = pad_sequences_X(X_test, max_len_X)

train_y = pad_sequences_y(y_train, max_len_y)
val_y   = pad_sequences_y(y_val, max_len_y)
test_y  = pad_sequences_y(y_test, max_len_y)

# datasets
train_dataset = TensorDataset(train_X, train_mask, train_y)
val_dataset   = TensorDataset(val_X, val_mask, val_y)
test_dataset  = TensorDataset(test_X, test_mask, test_y)

# loaders
batch_size = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

In [16]:
import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import copy
import random
import matplotlib.pyplot as plt
import unicodedata
from collections import Counter



from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
import Levenshtein as Lev

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

PAD_ID = 0
SOS_ID = 1
EOS_ID = 2


cuda


In [17]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

In [18]:
videos = os.path.join("..", "videos_all_languages.json")

with open(videos, "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.json_normalize(data["videos"])

print(df.columns)
df.head()

Index(['id', 'language', 'Sign', 'filepath', 'HamNoSys', 'concept_url',
       'HandLandmark', 'annotated_video'],
      dtype='object')


,id,language,Sign,filepath,HamNoSys,concept_url,HandLandmark,annotated_video
0,BSL_1,BSL,April,/ephemeral/Project/Data/External/BSL/April.mp4,,https://www.sign-lang.uni-hamburg.de/dicta-sig...,/ephemeral/Project/Data/Interim/BSL/April.json,/ephemeral/Project/Data/Interim/BSL/annotated/...
1,BSL_2,BSL,Athens,/ephemeral/Project/Data/External/BSL/Athens.mp4,,https://www.sign-lang.uni-hamburg.de/dicta-sig...,/ephemeral/Project/Data/Interim/BSL/Athens.json,/ephemeral/Project/Data/Interim/BSL/annotated/...
2,BSL_3,BSL,August,/ephemeral/Project/Data/External/BSL/August.mp4,,https://www.sign-lang.uni-hamburg.de/dicta-sig...,/ephemeral/Project/Data/Interim/BSL/August.json,/ephemeral/Project/Data/Interim/BSL/annotated/...
3,BSL_4,BSL,Berlin,/ephemeral/Project/Data/External/BSL/Berlin.mp4,,https://www.sign-lang.uni-hamburg.de/dicta-sig...,/ephemeral/Project/Data/Interim/BSL/Berlin.json,/ephemeral/Project/Data/Interim/BSL/annotated/...
4,BSL_5,BSL,February,/ephemeral/Project/Data/External/BSL/February.mp4,,https://www.sign-lang.uni-hamburg.de/dicta-sig...,/ephemeral/Project/Data/Interim/BSL/February.json,/ephemeral/Project/Data/Interim/BSL/annotated/...


In [19]:
# =========================
# CELL 3 — Get Landmark Paths
# =========================
Hand_Landmarks_paths = df["HandLandmark"]
Hand_Landmarks = [p for p in Hand_Landmarks_paths if os.path.exists(p)]

print("Valid landmark files:", len(Hand_Landmarks))

Valid landmark files: 3037


In [20]:
# =========================
# CELL 4 — Utility Functions
# =========================
def fix_mojibake(name):
    try:
        name = name.encode("latin1").decode("utf8")
    except:
        pass

    replacements = {
        "ÃŸ": "ß", "Ã„": "Ä", "Ã–": "Ö", "Ãœ": "Ü",
        "Ã¤": "ä", "Ã¶": "ö", "Ã¼": "ü",
        "Ã©": "é", "Ã¨": "è", "Ãª": "ê", "Ã«": "ë",
        "Ã´": "ô", "Ã¢": "â", "Ã¹": "ù", "Ã»": "û",
        "Ã§": "ç",
    }

    for bad, good in replacements.items():
        name = name.replace(bad, good)

    return unicodedata.normalize("NFC", name)


def normalize(seq):
    valid = np.any(seq != 0, axis=1)

    if valid.sum() < 2:
        return np.zeros_like(seq)

    center = seq[valid].mean(axis=0)
    seq[valid] -= center

    scale = np.std(seq[valid])
    if scale < 1e-6:
        return np.zeros_like(seq)

    seq[valid] /= scale
    return seq

In [21]:
HAND_LANDMARK_IDS = list(range(21))
POSE_LANDMARK_IDS = list(range(33))

REDUCED_FACE_LANDMARKS = (
    list(range(70, 103)) +
    list(range(0, 17)) +
    list(range(61, 88)) +
    list(range(152, 172))
)

In [22]:
def convert_frames_to_sequence(frames):
    rows = []

    for frame in frames:
        frame_features = []

        left = {
            lm["id"]: lm
            for hand in frame["hands"] if hand["handedness"] == "Left"
            for lm in hand["landmarks"]
        }

        right = {
            lm["id"]: lm
            for hand in frame["hands"] if hand["handedness"] == "Right"
            for lm in hand["landmarks"]
        }

        for i in HAND_LANDMARK_IDS:
            lm = left.get(i)
            frame_features.extend([0,0,0] if lm is None else [lm["x"], lm["y"], lm["z"]])

        for i in HAND_LANDMARK_IDS:
            lm = right.get(i)
            frame_features.extend([0,0,0] if lm is None else [lm["x"], lm["y"], lm["z"]])

        pose_dict = {lm["id"]: lm for p in frame["pose"] for lm in p["landmarks"]}
        for i in POSE_LANDMARK_IDS:
            lm = pose_dict.get(i)
            frame_features.extend([0,0,0] if lm is None else [lm["x"], lm["y"], lm["z"]])

        face_dict = {lm["id"]: lm for f in frame["face"] for lm in f["landmarks"]}
        for i in REDUCED_FACE_LANDMARKS:
            lm = face_dict.get(i)
            frame_features.extend([0,0,0] if lm is None else [lm["x"], lm["y"], lm["z"]])

        rows.append(np.array(frame_features, dtype=np.float32))

    return np.stack(rows)

In [23]:
# =========================
# CELL 8 — Build Label Dictionary
# =========================
df["filepath"] = df["filepath"].apply(
    lambda p: fix_mojibake(
        os.path.basename(p)
        .replace(".mp4", "")
        .replace(".MOV", "")
    )
)

label_dict = dict(zip(df["filepath"], df["HamNoSys"]))
print("Labels:", len(label_dict))

Labels: 2997


In [24]:
# =========================
# CELL 9 — Load X / y
# =========================
X = []
y = []

for seq_file in os.listdir("processed_sequences"):
    vid = seq_file.replace(".npy", "")
    clean_vid = fix_mojibake(vid)

    if clean_vid in label_dict:
        X.append(np.load(f"processed_sequences/{seq_file}"))
        y.append(label_dict[clean_vid])

print("Samples:", len(X))

Samples: 3055


In [25]:
# =========================
# CELL 7 — Convert JSON Files to .npy
# =========================
os.makedirs("processed_sequences", exist_ok=True)

X, y = [], []

for path in Hand_Landmarks:
    with open(path, "r") as f:
        data = json.load(f)

    vid = fix_mojibake(os.path.basename(data["video_path"]).replace(".mp4",""))

    seq = convert_frames_to_sequence(data["frames"])
    seq = normalize(seq)

    if vid in label_dict:
        X.append(seq)
        y.append(label_dict[vid])

In [26]:
# =========================
# TOKENISATION (FIXED)
# =========================

def tokenize_hamnosys(text):
    """
    Correct character-level tokenisation for HamNoSys.
    Keeps Unicode symbols as individual tokens.
    """
    return list(text.strip())


# build vocabulary
all_tokens = sorted({
    tok
    for seq in y
    for tok in tokenize_hamnosys(seq)
})

token_to_id = {t: i + 3 for i, t in enumerate(all_tokens)}
token_to_id["<PAD>"] = 0
token_to_id["<SOS>"] = 1
token_to_id["<EOS>"] = 2

id_to_token = {v: k for k, v in token_to_id.items()}


vocab_size = len(token_to_id)

BLANK_ID = vocab_size



# encode sequences
y_encoded = []

for seq in y:
    tokens = tokenize_hamnosys(seq)

    encoded = (
        [SOS_ID] +
        [token_to_id[t] for t in tokens] +
        [SOS_ID]
    )

    y_encoded.append(encoded)

In [27]:
def pad_sequences_X(seqs, max_len):
    feat = seqs[0].shape[1]
    X = torch.zeros(len(seqs), max_len, feat)
    mask = torch.zeros(len(seqs), max_len, dtype=torch.bool)

    for i, s in enumerate(seqs):
        l = min(len(s), max_len)
        X[i, :l] = torch.tensor(s[:l])
        mask[i, :l] = 1

    return X, mask


def pad_sequences_y(seqs, max_len):
    Y = torch.full((len(seqs), max_len), PAD_ID)

    for i, s in enumerate(seqs):
        l = min(len(s), max_len)
        Y[i, :l] = torch.tensor(s[:l])

    return Y

In [28]:
X_train, X_tmp, y_train, y_tmp = train_test_split(X, y_encoded, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.5, random_state=42)

In [29]:

print(np.__version__)
print(torch.__version__)
print(torch.from_numpy(np.array([1,2,3])))

1.26.4
2.5.1+cu121
tensor([1, 2, 3])


In [30]:
# =========================
# CELL 14 — Create Datasets / Loaders (NO *_tensor VARIABLES)
# =========================

max_len_X = max(len(s) for s in X_train + X_val + X_test)
max_len_y = max(len(s) for s in y_train + y_val + y_test)

# create padded tensors directly
train_X, train_mask = pad_sequences_X(X_train, max_len_X)
val_X, val_mask     = pad_sequences_X(X_val, max_len_X)
test_X, test_mask   = pad_sequences_X(X_test, max_len_X)

train_y = pad_sequences_y(y_train, max_len_y)
val_y   = pad_sequences_y(y_val, max_len_y)
test_y  = pad_sequences_y(y_test, max_len_y)

# datasets
train_dataset = TensorDataset(train_X, train_mask, train_y)
val_dataset   = TensorDataset(val_X, val_mask, val_y)
test_dataset  = TensorDataset(test_X, test_mask, test_y)

# loaders
batch_size = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

In [31]:


print("Using device:", device)
print(torch.cuda.is_available())
print(torch.cuda.device_count())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

Using device: cuda
True
1
NVIDIA H100 PCIe


In [32]:
def mild_noise(seq, noise_level=0.01):
    return seq + np.random.normal(0, noise_level, seq.shape)

In [33]:
def temporal_warp(seq, max_shift=3):
    T = seq.shape[0]

    if T < 10:
        return None

    shifts = np.random.randint(-max_shift, max_shift + 1, size=T)
    new_idx = np.clip(np.arange(T) + shifts, 0, T - 1)

    return seq[new_idx]


def motion_dropout(seq, drop_prob=0.05):
    mask = np.random.rand(seq.shape[0]) > drop_prob

    if mask.sum() < 10:
        return None

    return seq[mask]

In [34]:
def temporal_subsample(seq, rates=[2, 3], MIN_FRAMES=20):
    augmented = []

    for r in rates:
        new_seq = seq[::r]
        if new_seq.shape[0] >= MIN_FRAMES:
            augmented.append(new_seq)

    return augmented

In [35]:
X_aug = []
y_aug = []

MIN_FRAMES = 20
PAD_ID = PAD_ID

for x_seq, y_seq in zip(X_train, y_train):

    if x_seq.shape[0] >= MIN_FRAMES:
        X_aug.append(x_seq)
        y_aug.append(y_seq)

    # temporal subsampling
    for new_seq in temporal_subsample(x_seq):
        X_aug.append(new_seq)
        y_aug.append(y_seq)

    # motion dropout
    dropped = motion_dropout(x_seq)
    if dropped is not None:
        X_aug.append(dropped)
        y_aug.append(y_seq)

    # temporal warp (NEW IMPORTANT FIX)
    warped = temporal_warp(x_seq)
    if warped is not None:
        X_aug.append(warped)
        y_aug.append(y_seq)

In [36]:
X_train_tensor, X_train_mask = pad_sequences_X(X_aug, max_len_X)
X_val_tensor, X_val_mask = pad_sequences_X(X_val, max_len_X)
X_test_tensor, X_test_mask = pad_sequences_X(X_test, max_len_X)

y_train_tensor = pad_sequences_y(y_aug, max_len_y)
y_val_tensor   = pad_sequences_y(y_val, max_len_y)
y_test_tensor  = pad_sequences_y(y_test, max_len_y)

In [37]:
class LearnablePositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        self.pe = nn.Parameter(torch.zeros(1, max_len, d_model))
        nn.init.normal_(self.pe, std=0.02)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

In [38]:
class Encoder(nn.Module):
    def __init__(self, input_dim, hidden_dim=256):
        super().__init__()

        self.proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(0.1)
        )

        self.pos = LearnablePositionalEncoding(hidden_dim)

        layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=8,
            dim_feedforward=hidden_dim * 4,
            dropout=0.1,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )

        self.encoder = nn.TransformerEncoder(layer, num_layers=6)

        self.norm = nn.LayerNorm(hidden_dim)

    def forward(self, x, pad_mask):
        x = self.proj(x)
        x = self.pos(x)
        x = self.encoder(x, src_key_padding_mask=pad_mask)
        return self.norm(x)

In [39]:
class CustomTransformerDecoderLayer(nn.Module):
    def __init__(self, d_model, nhead, dropout=0.2):
        super().__init__()

        self.self_attn = nn.MultiheadAttention(
            d_model,
            nhead,
            dropout=dropout,
            batch_first=True
        )

        self.cross_attn = nn.MultiheadAttention(
            d_model,
            nhead,
            dropout=dropout,
            batch_first=True
        )

        self.linear1 = nn.Linear(d_model, d_model * 4)
        self.linear2 = nn.Linear(d_model * 4, d_model)

        self.dropout = nn.Dropout(dropout)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)
        
        
    def add_monotonic_bias(attn_scores):

            B, H, T, S = attn_scores.shape

            pos_t = torch.arange(T, device=attn_scores.device).unsqueeze(1)
            pos_s = torch.arange(S, device=attn_scores.device).unsqueeze(0)

            distance = torch.abs(pos_t - pos_s).float()

            bias = -distance * 0.1   # tuneable strength

            return attn_scores + bias
       


    def forward(
        self,
        tgt,
        memory,
        tgt_mask,
        tgt_key_padding_mask,
        memory_key_padding_mask
    ):

        # Self Attention
        _tgt, _ = self.self_attn(
            tgt,
            tgt,
            tgt,
            attn_mask=tgt_mask,
            key_padding_mask=tgt_key_padding_mask
        )

        tgt = self.norm1(tgt + self.dropout1(_tgt))

        # Cross Attention
        _tgt, cross_attn_weights = self.cross_attn(
            tgt,
            memory,
            memory,
            key_padding_mask=memory_key_padding_mask,
            need_weights=True,
            average_attn_weights=False
        )
        
        cross_attn_weights = cross_attn_weights / (cross_attn_weights.sum(dim=-1, keepdim=True) + 1e-6)

        tgt = self.norm2(tgt + self.dropout2(_tgt))

        # Feed Forward
        ff = self.linear2(
            self.dropout(
                F.relu(
                    self.linear1(tgt)
                )
            )
        )

        tgt = self.norm3(tgt + self.dropout3(ff))

        return tgt, cross_attn_weights

In [40]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, hidden_dim=256, num_layers=3):
        super().__init__()

        self.embed = nn.Embedding(vocab_size, hidden_dim)
        self.pos = LearnablePositionalEncoding(hidden_dim)

        self.layers = nn.ModuleList([
            CustomTransformerDecoderLayer(hidden_dim, 8)
            for _ in range(num_layers)
        ])

        self.fc = nn.Linear(hidden_dim, vocab_size + 1)

    def forward(
        self,
        tgt,
        memory,
        tgt_mask,
        tgt_pad_mask,
        src_pad_mask
    ):

        x = self.embed(tgt)
        x = self.pos(x)

        all_attn = []

        for layer in self.layers:
            x, attn = layer(
                x,
                memory,
                tgt_mask,
                tgt_pad_mask,
                src_pad_mask
            )
            all_attn.append(attn)

        logits = self.fc(x)

        return logits, all_attn[-1]


In [41]:
ce_loss_fn = nn.CrossEntropyLoss(ignore_index=PAD_ID)
ctc_loss_fn = nn.CTCLoss(blank=BLANK_ID, zero_infinity=True)

In [42]:
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID, label_smoothing=0.1)

In [43]:
def compute_coverage_loss(attn):

    # attn: [B, T, S]
    attn = attn.mean(dim=1)  # average heads → [B, T, S]

    B, T, S = attn.shape

    coverage = torch.zeros(B, S, device=attn.device)

    loss = 0.0

    for t in range(T):

        a_t = attn[:, t, :]  # [B, S]

        overlap = torch.min(a_t, coverage)

        loss += overlap.sum(dim=-1).mean()

        coverage = torch.clamp(coverage + a_t, max=1.0)

    return loss / (T + 1e-6)

In [44]:
def prepare_ctc_targets(batch):

    targets = []
    lengths = []

    for seq in batch:

        seq = [
            t for t in seq
            if t not in (SOS_ID, EOS_ID, PAD_ID, BLANK_ID)
        ]

        targets.append(torch.tensor(seq, dtype=torch.long))
        lengths.append(len(seq))

    targets = torch.cat(targets)

    return targets, lengths

In [45]:
class Seq2Seq(nn.Module):
    def __init__(
        self,
        input_dim,
        vocab_size,
        hidden_dim=256
    ):
        super().__init__()

        self.encoder = Encoder(input_dim, hidden_dim)

        self.decoder = Decoder(
            vocab_size=vocab_size,
            hidden_dim=hidden_dim
        )

        self.ctc_head = nn.Linear(hidden_dim, vocab_size + 1)

    def generate_square_subsequent_mask(self, sz, device=None):
        if device is None:
            device = next(self.parameters()).device

        return torch.triu(
            torch.ones(sz, sz, device=device),
            diagonal=1
        ).bool()

    def forward(self, src, src_mask, tgt):

        src_pad_mask = (src_mask == 0)
        tgt_pad_mask = (tgt == 0)

        memory = self.encoder(src, src_pad_mask)

        tgt_mask = self.generate_square_subsequent_mask(
            tgt.size(1),
            tgt.device
        )

        logits, attn = self.decoder(
            tgt,
            memory,
            tgt_mask,
            tgt_pad_mask,
            src_pad_mask
        )

        ctc_logits = self.ctc_head(memory)
        ctc_logits = F.log_softmax(ctc_logits, dim=-1)

        return logits, ctc_logits, attn

In [46]:
X_tensor = X_train_tensor
num_features = X_train_tensor.shape[2]
hidden_size = 256

model = Seq2Seq(
    input_dim=num_features,
    vocab_size=vocab_size,
    hidden_dim=hidden_size
).to(device)



/home/ubuntu/.local/lib/python3.10/site-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


In [47]:


def update_ema(model, ema, decay=0.999):
    with torch.no_grad():
        for p, e in zip(model.parameters(), ema.parameters()):
            e.data.mul_(decay).add_(p.data, alpha=1-decay)

In [48]:
def get_lr(
    epoch,
    base_lr=3e-4,
    warmup_epochs=3,
    max_epochs=100
):

    if epoch < warmup_epochs:
        return base_lr * ((epoch + 1) / warmup_epochs)

    progress = (
        (epoch - warmup_epochs) /
        (max_epochs - warmup_epochs)
    )

    return base_lr * 0.5 * (
        1 + math.cos(math.pi * progress)
    )

In [49]:
def clean(seq, SOS_ID, EOS_ID, PAD_ID):

    seq = [
        t for t in seq
        if t not in (SOS_ID, PAD_ID)
    ]

    if EOS_ID in seq:
        seq = seq[:seq.index(EOS_ID)]

    return seq

In [50]:
def length_penalty(seq, alpha):
    return ((5 + len(seq)) / 6) ** alpha


def eos_adjust(
    score,
    seq,
    EOS_ID,
    eos_bonus=1.0,
    eos_penalty=1.0
):
    if seq[-1] == EOS_ID:
        return score + eos_bonus

    return score - eos_penalty

In [51]:
def greedy_decode(model, src, src_mask, max_len=200):

    model.eval()

    src = src.to(device)
    src_mask = src_mask.to(device)

    with torch.no_grad():

        memory = model.encoder(src, (src_mask == 0))
        src_pad_mask = (src_mask == 0)

        seq = [SOS_ID]

        for _ in range(max_len):

            tgt = torch.tensor(seq, device=device).unsqueeze(0)

            tgt_mask = model.generate_square_subsequent_mask(len(seq), device)

            logits, _ = model.decoder(
                tgt,
                memory,
                tgt_mask,
                (tgt == PAD_ID),
                src_pad_mask
            )

            next_token = logits[:, -1].argmax(-1).item()

            seq.append(next_token)

            if next_token == EOS_ID:
                break

        return seq

In [52]:
print("SOS_ID:", SOS_ID)
print("EOS_ID:", EOS_ID)
print("PAD_ID:", PAD_ID)



SOS_ID: 1
EOS_ID: 2
PAD_ID: 0


In [53]:
def beam_decode(model, src, src_mask, beam_size=5, max_len=200, alpha=1.0):
    model.eval()

    src = src.to(device)
    src_mask = src_mask.to(device)

    with torch.no_grad():
        # encoder
        memory = model.encoder(src, (src_mask == 0).bool())
        src_pad_mask = (src_mask == 0)

        beams = [([SOS_ID], 0.0)]

        for _ in range(max_len):
            new_beams = []

            for seq, score in beams:

                # stop expansion if EOS
                if seq[-1] == EOS_ID:
                    new_beams.append((seq, score))
                    continue

                # target input
                tgt = torch.tensor(seq, device=device).unsqueeze(0)

                tgt_mask = model.generate_square_subsequent_mask(tgt.size(1)).to(device)
                tgt_pad_mask = (tgt == PAD_ID)

                # decoder forward
                logits, _ = model.decoder(
                    tgt,
                    memory,
                    tgt_mask,
                    tgt_pad_mask,
                    src_pad_mask
                )

                # FIX: always take last timestep properly
                logits = logits[:, -1, :]  # [1, vocab]

                log_probs = F.log_softmax(logits, dim=-1).squeeze(0)

                vocab_size = log_probs.size(-1)

                # repetition penalty (correct form)
                tok_counts = Counter(seq)

                for tok, c in tok_counts.items():
                    log_probs[tok] -= min(1.0, 0.2 * c)
                        

                topk = torch.topk(log_probs, beam_size)

                for i in range(beam_size):
                    token = topk.indices[i].item()
                    token_score = topk.values[i].item()

                    new_seq = seq + [token]
                    new_score = score + token_score

                    new_beams.append((new_seq, new_score))

            # length-normalized beam selection
            beams = sorted(
                new_beams,
                key=lambda x: x[1] / (((5 + len(x[0])) / 6) ** alpha),
                reverse=True
            )[:beam_size]

        return beams[0][0]

In [54]:
import heapq

def best_first_decode(model, src, src_mask, beam_size=5, max_len=200, alpha=1.0):
    model.eval()
    src = src.to(device)
    src_mask = src_mask.to(device)

    with torch.no_grad():
        # Encode + mask memory
        memory = model.encoder(src, (src_mask == 0))
        src_pad_mask = (src_mask == 0)

        # Priority queue: (negative_score, sequence)
        heap = [(0.0, [SOS_ID])]
        completed = []

        while heap:
            neg_score, seq = heapq.heappop(heap)
            score = -neg_score

            # If EOS or max length reached → finalize
            if seq[-1] == EOS_ID or len(seq) >= max_len:
                completed.append((seq, score + 1.0))  # EOS bonus
                if len(completed) >= beam_size:
                    break
                continue

            # Prepare decoder input
            tgt = torch.tensor(seq, device=device).unsqueeze(0)
            tgt_mask = model.generate_square_subsequent_mask(len(seq), device)

            logits, _ = model.decoder(
                tgt,
                memory,
                tgt_mask,
                (tgt == PAD_ID),
                (src_mask == 0)
            )

            # FORCE stable shape
            if logits.dim() == 3:
                logits = logits[:, -1, :]   # [1, V]
            elif logits.dim() == 2:
                logits = logits
            else:
                raise RuntimeError(f"Bad logits shape: {logits.shape}")

            log_probs = F.log_softmax(logits, dim=-1).squeeze(0)

            # repetition penalty
            tok_counts = Counter(seq)

            for tok, c in tok_counts.items():
                log_probs[tok] -= min(1.0, 0.2 * c)

            # expand top candidates
            topk = torch.topk(log_probs, beam_size)

            for i in range(beam_size):
                token = topk.indices[i].item()
                token_score = topk.values[i].item()

                new_seq = seq + [token]
                new_score = score + token_score

                heapq.heappush(heap, (-new_score, new_seq))

            # prune heap to keep search efficient
            if len(heap) > beam_size * 6:
                heap = heapq.nsmallest(beam_size * 3, heap)
                heapq.heapify(heap)

        # If we have completed sequences, pick best
        if completed:
            completed = sorted(
                completed,
                key=lambda x: x[1] / (((5 + len(x[0])) / 6) ** alpha),
                reverse=True
            )
            return completed[0][0]

        # fallback
        return heapq.heappop(heap)[1] if heap else [SOS_ID]


In [55]:
hidden_size = 256

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)

ema_model = copy.deepcopy(model).to(device)
ema_model.eval()
ema_decay = 0.999

In [56]:
def train_one_epoch(model, loader, optimizer, criterion,
                    use_ctc=False, use_coverage=False):

    model.train()
    total = 0

    for src, mask, tgt in loader:
        src, mask, tgt = src.to(device), mask.to(device), tgt.to(device)

        optimizer.zero_grad()

        inp = tgt[:, :-1]
        gold = tgt[:, 1:]

        logits, ctc_logits, attn = model(src, mask, inp)

        # ---------------- CE LOSS ----------------
        ce_loss = criterion(
            logits.reshape(-1, logits.size(-1)),
            gold.reshape(-1)
        )

        loss = ce_loss

        # ---------------- CTC LOSS ----------------
        if use_ctc:
            targets, target_lengths = prepare_ctc_targets(tgt.cpu().numpy())

            targets = targets.to(device)

            target_lengths = torch.tensor(
                target_lengths,
                device=device
            )

            input_lengths = torch.full(
                (src.size(0),),
                ctc_logits.size(1),
                dtype=torch.long,
                device=device
            )

            ctc_loss = ctc_loss_fn(
                ctc_logits.transpose(0, 1),
                targets,
                input_lengths,
                target_lengths
            )

            loss = loss + 0.3 * ctc_loss

        # ---------------- COVERAGE LOSS ----------------
        if use_coverage:
            cov_loss = compute_coverage_loss(attn)
            loss = loss + 0.1 * cov_loss

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()

        update_ema(model, ema_model)

        total += loss.item()

    return total / len(loader)

In [57]:
def save_checkpoint(model, ema_model, epoch, val_loss, path):
    torch.save({
        "epoch": epoch,
        "val_loss": val_loss,
        "model_state": model.state_dict(),
        "ema_state": ema_model.state_dict(),
    }, path)


In [58]:
num_epochs = 100
patience = 50

best_val_loss = float("inf")
best_cer = float("inf")
patience_counter = 0

os.makedirs("checkpoints", exist_ok=True)

print("Starting training...")

history = {
    "epoch": [],
    "train_loss": [],
    "val_loss": [],
    "cer_beam": [],
    "bleu": []
}

for epoch in range(num_epochs):

    lr = get_lr(epoch)
    for pg in optimizer.param_groups:
        pg["lr"] = lr

    train_loss = train_one_epoch(model, train_loader, optimizer, criterion)

    print(f"Epoch {epoch} | Train Loss: {train_loss}")

    history["epoch"].append(epoch)
    history["train_loss"].append(train_loss)
    
    ema_model.eval()

    with torch.no_grad():

        val_loss = 0.0

        for src, src_mask, tgt in val_loader:

            src = src.to(device)
            src_mask = src_mask.to(device)
            tgt = tgt.to(device)

            logits, _, _ = ema_model(src, src_mask, tgt[:, :-1])

            pred = logits.reshape(-1, logits.size(-1))
            gold = tgt[:, 1:].reshape(-1)

            mask = gold != PAD_ID

            loss = criterion(
                pred.reshape(-1, pred.size(-1)),
                gold.reshape(-1)
            )
            val_loss += loss.item()

    val_loss /= len(val_loader)
    

    print("Val Loss:", val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0

        save_checkpoint(
            model,
            ema_model,
            epoch,
            val_loss,
            "checkpoints/best.pt"
        )
    else:
        patience_counter += 1


    if patience_counter >= patience:
        print("Early stopping")
        
    
        break




Starting training...
Epoch 0 | Train Loss: 4.082807956116922
Val Loss: 5.1589792402167065
Epoch 1 | Train Loss: 3.259889195474346
Val Loss: 4.856536037043521
Epoch 2 | Train Loss: 2.987333595083001
Val Loss: 4.561866032449823
Epoch 3 | Train Loss: 2.823399629485741
Val Loss: 4.284850471898129
Epoch 4 | Train Loss: 2.7243663568175243
Val Loss: 4.028118083351536
Epoch 5 | Train Loss: 2.638955081446787
Val Loss: 3.7948433349007056
Epoch 6 | Train Loss: 2.582481740565782
Val Loss: 3.5878075925927413
Epoch 7 | Train Loss: 2.526500618859623
Val Loss: 3.407976690091585
Epoch 8 | Train Loss: 2.4667554866062122
Val Loss: 3.2551478712182296
Epoch 9 | Train Loss: 2.4213732762283153
Val Loss: 3.1271962994023372
Epoch 10 | Train Loss: 2.371380275554871
Val Loss: 3.0204969958255163
Epoch 11 | Train Loss: 2.3208019519120118
Val Loss: 2.9319519996643066
Epoch 12 | Train Loss: 2.272449329997716
Val Loss: 2.8589797019958496
Epoch 13 | Train Loss: 2.2362162156051464
Val Loss: 2.798375179893092
Epoch 14 |

KeyboardInterrupt: 

In [59]:
state = torch.load("checkpoints/best.pt", map_location=device)

print("Loaded checkpoint:")
print("  epoch:", state["epoch"])
print("  val_loss:", state["val_loss"])

model.load_state_dict(state["model_state"])
ema_model.load_state_dict(state["ema_state"])


Loaded checkpoint:
  epoch: 23
  val_loss: 2.602262911043669


/tmp/ipykernel_1499801/3704055925.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load("checkpoints/best.pt", map_location=device)


<All keys matched successfully>

In [60]:
def decode_and_clean(seq, id_to_token):

    seq = clean(seq, SOS_ID, EOS_ID, PAD_ID)

    return [id_to_token[t] for t in seq]

In [61]:
from nltk.translate.bleu_score import (
    sentence_bleu,
    SmoothingFunction
)

smooth = SmoothingFunction().method1


def compute_bleu(pred, true):

    return {
        "BLEU-1": sentence_bleu(
            [true],
            pred,
            weights=(1,0,0,0),
            smoothing_function=smooth
        ),

        "BLEU-2": sentence_bleu(
            [true],
            pred,
            weights=(0.5,0.5,0,0),
            smoothing_function=smooth
        ),

        "BLEU-4": sentence_bleu(
            [true],
            pred,
            weights=(0.25,0.25,0.25,0.25),
            smoothing_function=smooth
        ),
    }


def compute_cer(pred, true):

    pred_str = "".join(pred)
    true_str = "".join(true)

    if len(true_str) == 0:
        return 1.0 if len(pred_str) > 0 else 0.0

    return Lev.distance(
        pred_str,
        true_str
    ) / len(true_str)

In [62]:
def compute_metrics(preds, trues):

    bleu_scores = []
    cer_scores = []
    exact = 0

    for pred, true in zip(preds, trues):

        bleu_scores.append(
            compute_bleu(pred, true)
        )

        cer_scores.append(
            compute_cer(pred, true)
        )

        if pred == true:
            exact += 1

    return {
        "BLEU-1": np.mean(
            [b["BLEU-1"] for b in bleu_scores]
        ),

        "BLEU-2": np.mean(
            [b["BLEU-2"] for b in bleu_scores]
        ),

        "BLEU-4": np.mean(
            [b["BLEU-4"] for b in bleu_scores]
        ),

        "CER": np.mean(cer_scores),

        "EXACT": exact / len(preds)
    }

In [63]:
def evaluate(model, loader, decoding_fn):

    model.eval()

    preds = []
    trues = []

    with torch.no_grad():

        for src, src_mask, tgt in loader:

            src = src.to(device)
            src_mask = src_mask.to(device)
            tgt = tgt.to(device)

            for i in range(src.size(0)):

                pred_seq = decoding_fn(
                    model,
                    src[i].unsqueeze(0),
                    src_mask[i].unsqueeze(0)
                )

                true_seq = tgt[i].tolist()

                pred_seq = clean(
                    pred_seq,
                    SOS_ID,
                    EOS_ID,
                    PAD_ID
                )

                true_seq = clean(
                    true_seq,
                    SOS_ID,
                    EOS_ID,
                    PAD_ID
                )

                preds.append(pred_seq)
                trues.append(true_seq)

    decoded_preds = [
        [id_to_token[t] for t in p]
        for p in preds
    ]

    decoded_trues = [
        [id_to_token[t] for t in t_]
        for t_ in trues
    ]

    return compute_metrics(
        decoded_preds,
        decoded_trues
    )

In [ ]:
greedy_results = evaluate(
    ema_model,
    val_loader,
    greedy_decode
)

print("GREEDY:", greedy_results)


beam_results = evaluate(
    ema_model,
    val_loader,
    beam_decode
)

print("BEAM:", beam_results)


best_first_results = evaluate(
    ema_model,
    val_loader,
    best_first_decode
)

print("BEST-FIRST:", best_first_results)

GREEDY: {'BLEU-1': 0.06055368441605506, 'BLEU-2': 0.033316399091373454, 'BLEU-4': 0.011206374716236853, 'CER': 10.816387192398041, 'EXACT': 0.0}
BEAM: {'BLEU-1': 0.08311521429232417, 'BLEU-2': 0.04904399363696168, 'BLEU-4': 0.017139774307590318, 'CER': 10.663867703794432, 'EXACT': 0.0}


In [ ]:
state = torch.load("checkpoints/best.pt")
print(state["epoch"])


In [ ]:
state = torch.load("checkpoints/best.pt", map_location="cpu")
print(list(state["model_state"].keys()))


In [ ]:
greedy_results_test = evaluate(ema_model, test_loader, greedy_decode)
beam_results_test = evaluate(ema_model, test_loader, beam_decode)
best_first_results_test = evaluate(ema_model, test_loader, best_first_decode)

In [ ]:
print("vocab_size:", vocab_size)
print("num_tokens:", len(token_to_id))

print(model.decoder.fc)
print(model.decoder.fc.out_features)

history["val_loss"].append(val_loss)

history["cer_beam"].append(beam_results_test["CER"])
history["bleu"].append(beam_results_test["BLEU-4"])

In [ ]:
def show_examples_all_decoders(
    model,
    loader,
    n=10
):

    model.eval()

    decoders = {
        "GREEDY": greedy_decode,
        "BEAM": beam_decode,
        "BEST-FIRST": best_first_decode
    }

    examples = []

    with torch.no_grad():

        for src, src_mask, tgt in loader:

            src = src.to(device)
            tgt = tgt.to(device)

            for i in range(src.size(0)):

                # TRUE IDS (no decoding to string)
                true_ids = clean(
                    tgt[i].tolist(),
                    SOS_ID,
                    EOS_ID,
                    PAD_ID
                )

                preds = {}

                for name, fn in decoders.items():

                    pred_ids = fn(
                        model,
                        src[i].unsqueeze(0),
                        src_mask[i].unsqueeze(0)
                    )

                    pred_ids = clean(
                        pred_ids,
                        SOS_ID,
                        EOS_ID,
                        PAD_ID
                    )

                    preds[name] = pred_ids

                examples.append((true_ids, preds))

                if len(examples) >= n:
                    return examples

    return examples

In [ ]:
examples = show_examples_all_decoders(
    ema_model,
    val_loader,
    n=10
)

for i, (true_ids, preds) in enumerate(examples):

    print(f"\nExample {i+1}")

    print("TRUE IDS:       ", true_ids)
    print("GREEDY IDS:     ", preds["GREEDY"])
    print("BEAM IDS:       ", preds["BEAM"])
    print("BEST-FIRST IDS: ", preds["BEST-FIRST"])

In [ ]:
with torch.no_grad():

    src1 = X_val_tensor[0:1].to(device)
    src2 = X_val_tensor[1:2].to(device)
    
    src_mask1 = (X_val_mask[0:1].to(device) == 0)
    mem1 = model.encoder(src1, src_mask1)

    src_mask2 = (X_val_mask[1:2].to(device) == 0)
    mem2 = model.encoder(src2, src_mask2)

    diff = (mem1 - mem2).abs().mean()

    print("Encoder difference:", diff.item())

In [ ]:
with torch.no_grad():

    tgt = torch.tensor(
        [[SOS_ID]]
    ).to(device)

    tgt_mask = model.generate_square_subsequent_mask(
        tgt.size(1)
    ).to(device)

    tgt_pad_mask = (tgt == PAD_ID)
    src_pad_mask = (X_val_mask[0:1].to(device) == 0)

    logits, _ = model.decoder(
        tgt,
        mem1,
        tgt_mask,
        tgt_pad_mask,
        src_pad_mask
    )

   

In [ ]:
import shap

model.eval()

class WrappedModel(torch.nn.Module):

    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):

        batch = x.shape[0]

        tgt = torch.tensor(
            [[SOS_ID]] * batch
        ).to(x.device)

        src_mask = torch.ones(
            x.shape[:2],
            dtype=torch.bool
        ).to(x.device)

        logits, _ = self.model(
            x,
            src_mask,
            tgt
        )

        probs = torch.softmax(
            logits[:, -1, :],
            dim=-1
        )

        return probs

In [ ]:
plt.figure(figsize=(12,6))

plt.plot(history["epoch"], history["train_loss"], label="Train Loss")
plt.plot(history["epoch"], history["val_loss"], label="Val Loss")
plt.plot(history["epoch"], history["cer_beam"], label="CER Beam")
plt.plot(history["epoch"], history["bleu"], label="BLEU")

plt.legend()
plt.title("Training Progress Overview")
plt.xlabel("Epoch")

plt.show()

In [ ]:
plt.plot(history["epoch"], history["train_loss"], label="Train Loss")

plt.plot(
    history["epoch"][:len(history["val_loss"])],
    history["val_loss"],
    label="Val Loss"
)

In [ ]:
def encoder_stats(model, loader):

    model.eval()

    all_means = []
    all_stds = []

    with torch.no_grad():

        for src, src_mask, _ in loader:

            src = src.to(device)

            mem = model.encoder(src, (src_mask == 0))

            all_means.append(
                mem.mean().item()
            )

            all_stds.append(
                mem.std().item()
            )

    return np.mean(all_means), np.mean(all_stds)

In [ ]:
with torch.no_grad():
    src = X_val_tensor[0:1].to(device)
    mask = (X_val_mask[0:1].to(device) == 0)

    mem = model.encoder(src, mask)

    tgt = torch.tensor([[SOS_ID]]).to(device)

    logits, ctc_logits, attn = model(src, mask, tgt)



In [ ]:
from sklearn.manifold import TSNE

def plot_embedding_space(model, loader, n_batches=5):

    model.eval()

    feats = []

    with torch.no_grad():

        for i, (src, src_mask, _) in enumerate(loader):

            if i >= n_batches:
                break

            src = src.to(device)

            mem = model.encoder(src, (src_mask == 0))

            feats.append(F.normalize(mem.mean(dim=1), dim=-1).cpu().numpy())

    feats = np.concatenate(feats, axis=0)

    tsne = TSNE(n_components=2)

    proj = tsne.fit_transform(feats)

    plt.scatter(proj[:,0], proj[:,1])
    plt.title("Encoder Space Structure")
    plt.show()

In [ ]:
def attention_entropy(attn):

    p = attn.mean(dim=1)

    p = p / (p.sum(dim=-1, keepdim=True) + 1e-8)

    entropy = -(p * torch.log(p)).sum(dim=-1)

    return entropy.mean().item()

In [ ]:


def token_distribution(model, loader):

    model.eval()

    counter = Counter()

    with torch.no_grad():

        for src, src_mask, _ in loader:

            src = src.to(device)

            logits, _ = model(
                src,
                src_mask,
                tgt=torch.tensor([[SOS_ID]]).to(device)
            )

            preds = logits.argmax(-1).flatten().tolist()

            counter.update(preds)

    return counter

In [ ]:
def motion_energy(seq):

    return np.mean(
        np.linalg.norm(
            seq[1:] - seq[:-1],
            axis=1
        )
    )

In [ ]:

def ids_to_string(seq, id_to_token, SOS_ID, EOS_ID, PAD_ID):
    tokens = []
    for t in seq:
        if t in (SOS_ID, PAD_ID):
            continue
        if t == EOS_ID:
            break
        tokens.append(id_to_token[t])
    return "".join(tokens)

def compute_cer_for_loader(model, loader, id_to_token, SOS_ID, EOS_ID, PAD_ID, decode_fn):
    model.eval()
    total_dist = 0
    total_len = 0

    with torch.no_grad():
        for src, src_mask, tgt in loader:
            src = src.to(device)
            src_mask = src_mask.to(device)
            tgt = tgt.to(device)

            for i in range(src.size(0)):
                x = src[i:i+1]
                m = src_mask[i:i+1]
                y_true_ids = tgt[i].tolist()

                # decode (e.g. beam_decode or greedy_decode)
                y_pred_ids = decode_fn(model, x, m)

                true_str = ids_to_string(y_true_ids, id_to_token, SOS_ID, EOS_ID, PAD_ID)
                pred_str = ids_to_string(y_pred_ids, id_to_token, SOS_ID, EOS_ID, PAD_ID)

                if len(true_str) == 0:
                    continue

                dist = Lev.distance(pred_str, true_str)
                total_dist += dist
                total_len += len(true_str)

    cer = total_dist / total_len if total_len > 0 else 0.0
    return cer


In [ ]:
def evaluate_config(use_ctc=True, use_coverage=True, decode_mode="beam"):
    if decode_mode == "beam":
        decode_fn = beam_decode
    else:
        decode_fn = greedy_decode

    cer = compute_cer_for_loader(
        ema_model,
        test_loader,
        id_to_token,
        SOS_ID,
        EOS_ID,
        PAD_ID,
        decode_fn
    )

    return cer

In [ ]:
configs = [
    {"use_ctc": True,  "use_coverage": True,  "decode_mode": "beam"},
    {"use_ctc": False, "use_coverage": True,  "decode_mode": "beam"},
    {"use_ctc": True,  "use_coverage": False, "decode_mode": "beam"},
    {"use_ctc": False, "use_coverage": False, "decode_mode": "beam"},
]

for cfg in configs:
    cer = evaluate_config(**cfg)
    print(cfg, "CER:", cer)


In [ ]:
def sample_qualitative_examples(
    model,
    loader,
    id_to_token,
    SOS_ID,
    EOS_ID,
    PAD_ID,
    decode_fn,
    num_examples=10
):
    model.eval()
    examples = []

    with torch.no_grad():
        for src, src_mask, tgt in loader:
            src = src.to(device)
            src_mask = src_mask.to(device)
            tgt = tgt.to(device)

            for i in range(src.size(0)):
                x = src[i:i+1]
                m = src_mask[i:i+1]
                y_true_ids = tgt[i].tolist()

                y_pred_ids = decode_fn(model, x, m)

                true_str = ids_to_string(y_true_ids, id_to_token, SOS_ID, EOS_ID, PAD_ID)
                pred_str = ids_to_string(y_pred_ids, id_to_token, SOS_ID, EOS_ID, PAD_ID)

                examples.append((true_str, pred_str))

                if len(examples) >= num_examples:
                    return examples

    return examples

# usage
examples = sample_qualitative_examples(
    ema_model,
    test_loader,
    id_to_token,
    SOS_ID,
    EOS_ID,
    PAD_ID,
    beam_decode
)

for i, (gt, pred) in enumerate(examples, 1):
    print(f"Example {i}")
    print("GT:   ", gt)
    print("Pred: ", pred)
    print("-" * 40)


In [ ]:
def plot_attention_heatmap(attn, src_mask, tgt_len, src_len):
    # attn: [layers, heads, tgt_len, src_len] or [heads, tgt_len, src_len]
    import seaborn as sns

    # example: use last layer, average heads
    if attn.dim() == 4:
        attn = attn[-1].mean(0)  # [tgt_len, src_len]

    attn = attn[:tgt_len, :src_len].cpu().numpy()

    plt.figure(figsize=(8, 6))
    sns.heatmap(attn, cmap="viridis")
    plt.xlabel("Source frames")
    plt.ylabel("Target positions")
    plt.title("Cross-attention")
    plt.show()


In [ ]:
def evaluate_with_perturbation(
    model,
    loader,
    id_to_token,
    SOS_ID,
    EOS_ID,
    PAD_ID,
    decode_fn,
    perturb_fn,
    label="perturbation"
):
    model.eval()
    total_dist = 0
    total_len = 0

    with torch.no_grad():
        for src, src_mask, tgt in loader:
            src = src.cpu().numpy()   # apply numpy-based perturbations
            src_mask = src_mask.numpy()
            tgt = tgt.to(device)

            for i in range(src.shape[0]):
                x_seq = src[i]  # [T, D]
                m_seq = src_mask[i]

                x_pert = perturb_fn(x_seq)  # must return [T', D]
                if x_pert is None:
                    continue

                T_new = x_pert.shape[0]
                x_tensor = torch.from_numpy(x_pert).unsqueeze(0).to(device)
                m_tensor = torch.ones(1, T_new, dtype=torch.bool, device=device)

                y_true_ids = tgt[i].tolist()
                y_pred_ids = decode_fn(model, x_tensor, m_tensor)

                true_str = ids_to_string(y_true_ids, id_to_token, SOS_ID, EOS_ID, PAD_ID)
                pred_str = ids_to_string(y_pred_ids, id_to_token, SOS_ID, EOS_ID, PAD_ID)

                if len(true_str) == 0:
                    continue

                dist = Lev.distance(pred_str, true_str)
                total_dist += dist
                total_len += len(true_str)

    cer = total_dist / total_len if total_len > 0 else 0.0
    print(f"[{label}] CER: {cer:.4f}")
    return cer


def perturb_subsample(seq):
    return seq[::2] if seq.shape[0] >= 20 else None

def perturb_dropout(seq):
    return motion_dropout(seq, drop_prob=0.1)

def perturb_warp(seq):
    return temporal_warp(seq)

evaluate_with_perturbation(
    ema_model,
    test_loader,
    id_to_token,
    SOS_ID,
    EOS_ID,
    PAD_ID,
    beam_decode,
    perturb_subsample,
    label="subsample x2"
)


In [ ]:
import time

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def benchmark_inference(
    model,
    loader,
    decode_fn,
    num_batches=10
):
    model.eval()
    start = time.time()
    n_samples = 0

    with torch.no_grad():
        for b, (src, src_mask, _) in enumerate(loader):
            if b >= num_batches:
                break
            src = src.to(device)
            src_mask = src_mask.to(device)

            for i in range(src.size(0)):
                x = src[i:i+1]
                m = src_mask[i:i+1]
                _ = decode_fn(model, x, m)
                n_samples += 1

    elapsed = time.time() - start
    if n_samples == 0:
        return None

    avg_time = elapsed / n_samples
    print(f"Inference: {avg_time:.4f} s/sample ({1.0/avg_time:.2f} samples/s)")
    return avg_time

# usage
n_params = count_parameters(model)
print("Trainable parameters:", n_params)

benchmark_inference(
    ema_model,
    test_loader,
    beam_decode,
    num_batches=5
)


In [ ]:
"""
Key observations:

- BLEU is computed using NLTK sentence_bleu
- CER uses Levenshtein distance normalized by length
- EXACT match is strict sequence equality
- Beam search improves accuracy but reduces diversity
- Encoder representation drift should stay stable
- Attention entropy can indicate overconfidence collapse

Potential improvements:

1. Relative positional encoding
2. Stronger data augmentation (temporal warp)
3. Label smoothing tuning
4. Better coverage penalty scheduling
5. Nucleus sampling decoding
6. Multi-signer conditioning (reduce bias)
"""

In [ ]:


print("Using device:", device)
print(torch.cuda.is_available())
print(torch.cuda.device_count())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
def mild_noise(seq, noise_level=0.01):
    return seq + np.random.normal(0, noise_level, seq.shape)

In [ ]:
def temporal_warp(seq, max_shift=3):
    T = seq.shape[0]

    if T < 10:
        return None

    shifts = np.random.randint(-max_shift, max_shift + 1, size=T)
    new_idx = np.clip(np.arange(T) + shifts, 0, T - 1)

    return seq[new_idx]


def motion_dropout(seq, drop_prob=0.05):
    mask = np.random.rand(seq.shape[0]) > drop_prob

    if mask.sum() < 10:
        return None

    return seq[mask]

In [ ]:
def temporal_subsample(seq, rates=[2, 3], MIN_FRAMES=20):
    augmented = []

    for r in rates:
        new_seq = seq[::r]
        if new_seq.shape[0] >= MIN_FRAMES:
            augmented.append(new_seq)

    return augmented

In [ ]:
X_aug = []
y_aug = []

MIN_FRAMES = 20
PAD_ID = PAD_ID

for x_seq, y_seq in zip(X_train, y_train):

    if x_seq.shape[0] >= MIN_FRAMES:
        X_aug.append(x_seq)
        y_aug.append(y_seq)

    # temporal subsampling
    for new_seq in temporal_subsample(x_seq):
        X_aug.append(new_seq)
        y_aug.append(y_seq)

    # motion dropout
    dropped = motion_dropout(x_seq)
    if dropped is not None:
        X_aug.append(dropped)
        y_aug.append(y_seq)

    # temporal warp (NEW IMPORTANT FIX)
    warped = temporal_warp(x_seq)
    if warped is not None:
        X_aug.append(warped)
        y_aug.append(y_seq)

In [ ]:
X_train_tensor, X_train_mask = pad_sequences_X(X_aug, max_len_X)
X_val_tensor, X_val_mask = pad_sequences_X(X_val, max_len_X)
X_test_tensor, X_test_mask = pad_sequences_X(X_test, max_len_X)

y_train_tensor = pad_sequences_y(y_aug, max_len_y)
y_val_tensor   = pad_sequences_y(y_val, max_len_y)
y_test_tensor  = pad_sequences_y(y_test, max_len_y)

In [ ]:
class LearnablePositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        self.pe = nn.Parameter(torch.zeros(1, max_len, d_model))
        nn.init.normal_(self.pe, std=0.02)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()

        self.proj = nn.Linear(input_dim, hidden_dim)
        self.pos = LearnablePositionalEncoding(hidden_dim)

        layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=4,
            batch_first=True,
            activation="gelu"
        )

        self.encoder = nn.TransformerEncoder(layer, num_layers=4)
        self.norm = nn.LayerNorm(hidden_dim)

    def forward(self, x, mask=None):
        if mask is None:
            mask = torch.zeros(x.size(0), x.size(1), device=x.device).bool()

        x = self.proj(x)
        x = self.pos(x)

        x = self.encoder(x, src_key_padding_mask=mask)

        return self.norm(x)

In [ ]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, hidden_dim, num_layers=2):
        super().__init__()

        self.embed = nn.Embedding(vocab_size, hidden_dim)
        self.pos = LearnablePositionalEncoding(hidden_dim)

        self.layers = nn.ModuleList([
            CustomTransformerDecoderLayer(hidden_dim, 4)
            for _ in range(num_layers)
        ])

        self.fc = nn.Linear(hidden_dim, vocab_size + 1)

    def forward(self, tgt, memory, tgt_mask, tgt_pad_mask, src_pad_mask):

        x = self.embed(tgt)
        x = self.pos(x)

        attn = None

        for layer in self.layers:
            x, attn = layer(
                x,
                memory,
                tgt_mask,
                tgt_pad_mask,
                src_pad_mask
            )

        logits = self.fc(x)

        return logits, attn

In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID, label_smoothing=0.1)

In [ ]:
def compute_coverage_loss(attn):
    # [B,H,T,S]
    attn = attn.mean(dim=1)

    B, T, S = attn.shape

    cov = torch.zeros(B, S, device=attn.device)

    loss = 0

    for t in range(T):
        step = attn[:, t]
        loss += torch.min(step, cov).sum(dim=-1).mean()
        cov = cov + step

    return loss / T

In [ ]:
def prepare_ctc_targets(batch):

    targets = []
    lengths = []

    for seq in batch:

        seq = [
            t for t in seq
            if t not in (SOS_ID, EOS_ID, PAD_ID, BLANK_ID)
        ]

        targets.append(torch.tensor(seq, dtype=torch.long))
        lengths.append(len(seq))

    targets = torch.cat(targets)

    return targets, lengths

In [ ]:
X_tensor = X_train_tensor
num_features = X_train_tensor.shape[2]
hidden_size = 256

model = Seq2Seq(
    input_dim=num_features,
    vocab_size=vocab_size,
    hidden_dim=hidden_size
).to(device)



In [ ]:


def update_ema(model, ema, decay=0.999):
    with torch.no_grad():
        for p, e in zip(model.parameters(), ema.parameters()):
            e.data.mul_(decay).add_(p.data, alpha=1-decay)

In [ ]:
def get_lr(
    epoch,
    base_lr=3e-4,
    warmup_epochs=3,
    max_epochs=100
):

    if epoch < warmup_epochs:
        return base_lr * ((epoch + 1) / warmup_epochs)

    progress = (
        (epoch - warmup_epochs) /
        (max_epochs - warmup_epochs)
    )

    return base_lr * 0.5 * (
        1 + math.cos(math.pi * progress)
    )

In [ ]:
def clean(seq, SOS_ID, EOS_ID, PAD_ID):

    seq = [
        t for t in seq
        if t not in (SOS_ID, PAD_ID)
    ]

    if EOS_ID in seq:
        seq = seq[:seq.index(EOS_ID)]

    return seq

In [ ]:
def length_penalty(seq, alpha):
    return ((5 + len(seq)) / 6) ** alpha


def eos_adjust(
    score,
    seq,
    EOS_ID,
    eos_bonus=1.0,
    eos_penalty=1.0
):
    if seq[-1] == EOS_ID:
        return score + eos_bonus

    return score - eos_penalty

In [ ]:
def greedy_decode(model, src, src_mask, max_len=200):

    model.eval()

    src = src.to(device)
    src_mask = src_mask.to(device)

    with torch.no_grad():

        memory = model.encoder(src, (src_mask == 0))
        src_pad_mask = (src_mask == 0)

        seq = [SOS_ID]

        for _ in range(max_len):

            tgt = torch.tensor(seq, device=device).unsqueeze(0)

            tgt_mask = model.generate_square_subsequent_mask(len(seq), device)

            logits, _ = model.decoder(
                tgt,
                memory,
                tgt_mask,
                (tgt == PAD_ID),
                src_pad_mask
            )

            next_token = logits[:, -1].argmax(-1).item()

            seq.append(next_token)

            if next_token == EOS_ID:
                break

        return seq

In [ ]:
print("SOS_ID:", SOS_ID)
print("EOS_ID:", EOS_ID)
print("PAD_ID:", PAD_ID)



In [ ]:
def beam_decode(model, src, src_mask, beam_size=5, max_len=200, alpha=1.0):
    model.eval()

    src = src.to(device)
    src_mask = src_mask.to(device)

    with torch.no_grad():
        # encoder
        memory = model.encoder(src, (src_mask == 0).bool())
        src_pad_mask = (src_mask == 0)

        beams = [([SOS_ID], 0.0)]

        for _ in range(max_len):
            new_beams = []

            for seq, score in beams:

                # stop expansion if EOS
                if seq[-1] == EOS_ID:
                    new_beams.append((seq, score))
                    continue

                # target input
                tgt = torch.tensor(seq, device=device).unsqueeze(0)

                tgt_mask = model.generate_square_subsequent_mask(tgt.size(1)).to(device)
                tgt_pad_mask = (tgt == PAD_ID)

                # decoder forward
                logits, _ = model.decoder(
                    tgt,
                    memory,
                    tgt_mask,
                    tgt_pad_mask,
                    src_pad_mask
                )

                # FIX: always take last timestep properly
                logits = logits[:, -1, :]  # [1, vocab]

                log_probs = F.log_softmax(logits, dim=-1).squeeze(0)

                vocab_size = log_probs.size(-1)

                # repetition penalty (correct form)
                tok_counts = Counter(seq)

                for tok, c in tok_counts.items():
                    log_probs[tok] -= min(1.0, 0.2 * c)
                        

                topk = torch.topk(log_probs, beam_size)

                for i in range(beam_size):
                    token = topk.indices[i].item()
                    token_score = topk.values[i].item()

                    new_seq = seq + [token]
                    new_score = score + token_score

                    new_beams.append((new_seq, new_score))

            # length-normalized beam selection
            beams = sorted(
                new_beams,
                key=lambda x: x[1] / (((5 + len(x[0])) / 6) ** alpha),
                reverse=True
            )[:beam_size]

        return beams[0][0]

In [ ]:
import heapq

def best_first_decode(model, src, src_mask, beam_size=5, max_len=200, alpha=1.0):
    model.eval()
    src = src.to(device)
    src_mask = src_mask.to(device)

    with torch.no_grad():
        # Encode + mask memory
        memory = model.encoder(src, (src_mask == 0))
        src_pad_mask = (src_mask == 0)

        # Priority queue: (negative_score, sequence)
        heap = [(0.0, [SOS_ID])]
        completed = []

        while heap:
            neg_score, seq = heapq.heappop(heap)
            score = -neg_score

            # If EOS or max length reached → finalize
            if seq[-1] == EOS_ID or len(seq) >= max_len:
                completed.append((seq, score + 1.0))  # EOS bonus
                if len(completed) >= beam_size:
                    break
                continue

            # Prepare decoder input
            tgt = torch.tensor(seq, device=device).unsqueeze(0)
            tgt_mask = model.generate_square_subsequent_mask(len(seq), device)

            logits, _ = model.decoder(
                tgt,
                memory,
                tgt_mask,
                (tgt == PAD_ID),
                (src_mask == 0)
            )

            # FORCE stable shape
            if logits.dim() == 3:
                logits = logits[:, -1, :]   # [1, V]
            elif logits.dim() == 2:
                logits = logits
            else:
                raise RuntimeError(f"Bad logits shape: {logits.shape}")

            log_probs = F.log_softmax(logits, dim=-1).squeeze(0)

            # repetition penalty
            tok_counts = Counter(seq)

            for tok, c in tok_counts.items():
                log_probs[tok] -= min(1.0, 0.2 * c)

            # expand top candidates
            topk = torch.topk(log_probs, beam_size)

            for i in range(beam_size):
                token = topk.indices[i].item()
                token_score = topk.values[i].item()

                new_seq = seq + [token]
                new_score = score + token_score

                heapq.heappush(heap, (-new_score, new_seq))

            # prune heap to keep search efficient
            if len(heap) > beam_size * 6:
                heap = heapq.nsmallest(beam_size * 3, heap)
                heapq.heapify(heap)

        # If we have completed sequences, pick best
        if completed:
            completed = sorted(
                completed,
                key=lambda x: x[1] / (((5 + len(x[0])) / 6) ** alpha),
                reverse=True
            )
            return completed[0][0]

        # fallback
        return heapq.heappop(heap)[1] if heap else [SOS_ID]


In [ ]:
hidden_size = 256

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)

ema_model = copy.deepcopy(model).to(device)
ema_model.eval()
ema_decay = 0.999

In [ ]:
num_epochs = 100
patience = 50

best_val_loss = float("inf")
best_cer = float("inf")
patience_counter = 0

os.makedirs("checkpoints", exist_ok=True)

print("Starting training...")

history = {
    "epoch": [],
    "train_loss": [],
    "val_loss": [],
    "cer_beam": [],
    "bleu": []
}

for epoch in range(num_epochs):

    lr = get_lr(epoch)
    for pg in optimizer.param_groups:
        pg["lr"] = lr

    train_loss = train_one_epoch(model, train_loader, optimizer, criterion)

    print(f"Epoch {epoch} | Train Loss: {train_loss}")

    history["epoch"].append(epoch)
    history["train_loss"].append(train_loss)
    
    ema_model.eval()

    with torch.no_grad():

        val_loss = 0.0

        for src, src_mask, tgt in val_loader:

            src = src.to(device)
            src_mask = src_mask.to(device)
            tgt = tgt.to(device)

            logits, _, _ = ema_model(src, src_mask, tgt[:, :-1])

            pred = logits.reshape(-1, logits.size(-1))
            gold = tgt[:, 1:].reshape(-1)

            mask = gold != PAD_ID

            loss = criterion(
                pred.reshape(-1, pred.size(-1)),
                gold.reshape(-1)
            )
            val_loss += loss.item()

    val_loss /= len(val_loader)
    

    print("Val Loss:", val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0

        save_checkpoint(
            model,
            ema_model,
            epoch,
            val_loss,
            "checkpoints/best.pt"
        )
    else:
        patience_counter += 1


    if patience_counter >= patience:
        print("Early stopping")
        break

save_checkpoint(
    model,
    ema_model,
    epoch,
    val_loss,
    "checkpoints/last.pt"
)


In [ ]:
print(model.encoder.encoder.num_layers)

In [ ]:
state = torch.load("checkpoints/best.pt", map_location=device)

print("Loaded checkpoint:")
print("  epoch:", state["epoch"])
print("  val_loss:", state["val_loss"])

model.load_state_dict(state["model_state"])
ema_model.load_state_dict(state["ema_state"])


In [ ]:
def decode_and_clean(seq, id_to_token):

    seq = clean(seq, SOS_ID, EOS_ID, PAD_ID)

    return [id_to_token[t] for t in seq]

In [ ]:
from nltk.translate.bleu_score import (
    sentence_bleu,
    SmoothingFunction
)

smooth = SmoothingFunction().method1


def compute_bleu(pred, true):

    return {
        "BLEU-1": sentence_bleu(
            [true],
            pred,
            weights=(1,0,0,0),
            smoothing_function=smooth
        ),

        "BLEU-2": sentence_bleu(
            [true],
            pred,
            weights=(0.5,0.5,0,0),
            smoothing_function=smooth
        ),

        "BLEU-4": sentence_bleu(
            [true],
            pred,
            weights=(0.25,0.25,0.25,0.25),
            smoothing_function=smooth
        ),
    }


def compute_cer(pred, true):

    pred_str = "".join(pred)
    true_str = "".join(true)

    if len(true_str) == 0:
        return 1.0 if len(pred_str) > 0 else 0.0

    return Lev.distance(
        pred_str,
        true_str
    ) / len(true_str)

In [ ]:
def compute_metrics(preds, trues):

    bleu_scores = []
    cer_scores = []
    exact = 0

    for pred, true in zip(preds, trues):

        bleu_scores.append(
            compute_bleu(pred, true)
        )

        cer_scores.append(
            compute_cer(pred, true)
        )

        if pred == true:
            exact += 1

    return {
        "BLEU-1": np.mean(
            [b["BLEU-1"] for b in bleu_scores]
        ),

        "BLEU-2": np.mean(
            [b["BLEU-2"] for b in bleu_scores]
        ),

        "BLEU-4": np.mean(
            [b["BLEU-4"] for b in bleu_scores]
        ),

        "CER": np.mean(cer_scores),

        "EXACT": exact / len(preds)
    }

In [ ]:
def evaluate(model, loader, decoding_fn):

    model.eval()

    preds = []
    trues = []

    with torch.no_grad():

        for src, src_mask, tgt in loader:

            src = src.to(device)
            src_mask = src_mask.to(device)
            tgt = tgt.to(device)

            for i in range(src.size(0)):

                pred_seq = decoding_fn(
                    model,
                    src[i].unsqueeze(0),
                    src_mask[i].unsqueeze(0)
                )

                true_seq = tgt[i].tolist()

                pred_seq = clean(
                    pred_seq,
                    SOS_ID,
                    EOS_ID,
                    PAD_ID
                )

                true_seq = clean(
                    true_seq,
                    SOS_ID,
                    EOS_ID,
                    PAD_ID
                )

                preds.append(pred_seq)
                trues.append(true_seq)

    decoded_preds = [
        [id_to_token[t] for t in p]
        for p in preds
    ]

    decoded_trues = [
        [id_to_token[t] for t in t_]
        for t_ in trues
    ]

    return compute_metrics(
        decoded_preds,
        decoded_trues
    )

In [ ]:
greedy_results = evaluate(
    ema_model,
    val_loader,
    greedy_decode
)

print("GREEDY:", greedy_results)


beam_results = evaluate(
    ema_model,
    val_loader,
    beam_decode
)

print("BEAM:", beam_results)


best_first_results = evaluate(
    ema_model,
    val_loader,
    best_first_decode
)

print("BEST-FIRST:", best_first_results)

In [ ]:
greedy_results_test = evaluate(ema_model, test_loader, greedy_decode)
beam_results_test = evaluate(ema_model, test_loader, beam_decode)
best_first_results_test = evaluate(ema_model, test_loader, best_first_decode)

In [ ]:
print("vocab_size:", vocab_size)
print("num_tokens:", len(token_to_id))

print(model.decoder.fc)
print(model.decoder.fc.out_features)

history["val_loss"].append(val_loss)

history["cer_beam"].append(beam_results_test["CER"])
history["bleu"].append(beam_results_test["BLEU-4"])

In [ ]:
def show_examples_all_decoders(
    model,
    loader,
    n=10
):

    model.eval()

    decoders = {
        "GREEDY": greedy_decode,
        "BEAM": beam_decode,
        "BEST-FIRST": best_first_decode
    }

    examples = []

    with torch.no_grad():

        for src, src_mask, tgt in loader:

            src = src.to(device)
            tgt = tgt.to(device)

            for i in range(src.size(0)):

                # TRUE IDS (no decoding to string)
                true_ids = clean(
                    tgt[i].tolist(),
                    SOS_ID,
                    EOS_ID,
                    PAD_ID
                )

                preds = {}

                for name, fn in decoders.items():

                    pred_ids = fn(
                        model,
                        src[i].unsqueeze(0),
                        src_mask[i].unsqueeze(0)
                    )

                    pred_ids = clean(
                        pred_ids,
                        SOS_ID,
                        EOS_ID,
                        PAD_ID
                    )

                    preds[name] = pred_ids

                examples.append((true_ids, preds))

                if len(examples) >= n:
                    return examples

    return examples

In [ ]:
examples = show_examples_all_decoders(
    ema_model,
    val_loader,
    n=10
)

for i, (true_ids, preds) in enumerate(examples):

    print(f"\nExample {i+1}")

    print("TRUE IDS:       ", true_ids)
    print("GREEDY IDS:     ", preds["GREEDY"])
    print("BEAM IDS:       ", preds["BEAM"])
    print("BEST-FIRST IDS: ", preds["BEST-FIRST"])

In [ ]:
with torch.no_grad():

    src1 = X_val_tensor[0:1].to(device)
    src2 = X_val_tensor[1:2].to(device)
    
    src_mask1 = (X_val_mask[0:1].to(device) == 0)
    mem1 = model.encoder(src1, src_mask1)

    src_mask2 = (X_val_mask[1:2].to(device) == 0)
    mem2 = model.encoder(src2, src_mask2)

    diff = (mem1 - mem2).abs().mean()

    print("Encoder difference:", diff.item())

In [ ]:
with torch.no_grad():

    tgt = torch.tensor(
        [[SOS_ID]]
    ).to(device)

    tgt_mask = model.generate_square_subsequent_mask(
        tgt.size(1)
    ).to(device)

    tgt_pad_mask = (tgt == PAD_ID)
    src_pad_mask = (X_val_mask[0:1].to(device) == 0)

    logits, _ = model.decoder(
        tgt,
        mem1,
        tgt_mask,
        tgt_pad_mask,
        src_pad_mask
    )

   

In [ ]:
import shap

model.eval()

class WrappedModel(torch.nn.Module):

    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):

        batch = x.shape[0]

        tgt = torch.tensor(
            [[SOS_ID]] * batch
        ).to(x.device)

        src_mask = torch.ones(
            x.shape[:2],
            dtype=torch.bool
        ).to(x.device)

        logits, _ = self.model(
            x,
            src_mask,
            tgt
        )

        probs = torch.softmax(
            logits[:, -1, :],
            dim=-1
        )

        return probs

In [ ]:
plt.figure(figsize=(12,6))

plt.plot(history["epoch"], history["train_loss"], label="Train Loss")
plt.plot(history["epoch"], history["val_loss"], label="Val Loss")
plt.plot(history["epoch"], history["cer_beam"], label="CER Beam")
plt.plot(history["epoch"], history["bleu"], label="BLEU")

plt.legend()
plt.title("Training Progress Overview")
plt.xlabel("Epoch")

plt.show()

In [ ]:
plt.plot(history["epoch"], history["train_loss"], label="Train Loss")

plt.plot(
    history["epoch"][:len(history["val_loss"])],
    history["val_loss"],
    label="Val Loss"
)

In [ ]:
def encoder_stats(model, loader):

    model.eval()

    all_means = []
    all_stds = []

    with torch.no_grad():

        for src, src_mask, _ in loader:

            src = src.to(device)

            mem = model.encoder(src, (src_mask == 0))

            all_means.append(
                mem.mean().item()
            )

            all_stds.append(
                mem.std().item()
            )

    return np.mean(all_means), np.mean(all_stds)

In [ ]:
with torch.no_grad():
    src = X_val_tensor[0:1].to(device)
    mask = (X_val_mask[0:1].to(device) == 0)

    mem = model.encoder(src, (src_mask == 0))

    tgt = torch.tensor([[SOS_ID]]).to(device)

    logits, ctc_logits, attn = model(src, mask, tgt)



In [ ]:
from sklearn.manifold import TSNE

def plot_embedding_space(model, loader, n_batches=5):

    model.eval()

    feats = []

    with torch.no_grad():

        for i, (src, src_mask, _) in enumerate(loader):

            if i >= n_batches:
                break

            src = src.to(device)

            mem = model.encoder(src, (src_mask == 0))

            feats.append(F.normalize(mem.mean(dim=1), dim=-1).cpu().numpy())

    feats = np.concatenate(feats, axis=0)

    tsne = TSNE(n_components=2)

    proj = tsne.fit_transform(feats)

    plt.scatter(proj[:,0], proj[:,1])
    plt.title("Encoder Space Structure")
    plt.show()

In [ ]:
def attention_entropy(attn):

    p = attn.mean(dim=1)

    p  = p / (p.sum(dim=-1, keepdim=True) + 1e-8)

    entropy = -(p * torch.log(p)).sum(dim=-1)

    return entropy.mean().item()

In [ ]:


def token_distribution(model, loader):

    model.eval()

    counter = Counter()

    with torch.no_grad():

        for src, src_mask, _ in loader:

            src = src.to(device)

            logits, _ = model(
                src,
                src_mask,
                tgt=torch.tensor([[SOS_ID]]).to(device)
            )

            preds = logits.argmax(-1).flatten().tolist()

            counter.update(preds)

    return counter

In [ ]:
def motion_energy(seq):

    return np.mean(
        np.linalg.norm(
            seq[1:] - seq[:-1],
            axis=1
        )
    )

In [ ]:

def ids_to_string(seq, id_to_token, SOS_ID, EOS_ID, PAD_ID):
    tokens = []
    for t in seq:
        if t in (SOS_ID, PAD_ID):
            continue
        if t == EOS_ID:
            break
        tokens.append(id_to_token[t])
    return "".join(tokens)

def compute_cer_for_loader(model, loader, id_to_token, SOS_ID, EOS_ID, PAD_ID, decode_fn):
    model.eval()
    total_dist = 0
    total_len = 0

    with torch.no_grad():
        for src, src_mask, tgt in loader:
            src = src.to(device)
            src_mask = src_mask.to(device)
            tgt = tgt.to(device)

            for i in range(src.size(0)):
                x = src[i:i+1]
                m = src_mask[i:i+1]
                y_true_ids = tgt[i].tolist()

                # decode (e.g. beam_decode or greedy_decode)
                y_pred_ids = decode_fn(model, x, m)

                true_str = ids_to_string(y_true_ids, id_to_token, SOS_ID, EOS_ID, PAD_ID)
                pred_str = ids_to_string(y_pred_ids, id_to_token, SOS_ID, EOS_ID, PAD_ID)

                if len(true_str) == 0:
                    continue

                dist = Lev.distance(pred_str, true_str)
                total_dist += dist
                total_len += len(true_str)

    cer = total_dist / total_len if total_len > 0 else 0.0
    return cer


In [ ]:
def evaluate_config(use_ctc=True, use_coverage=True, decode_mode="beam"):
    if decode_mode == "beam":
        decode_fn = beam_decode
    else:
        decode_fn = greedy_decode

    cer = compute_cer_for_loader(
        ema_model,
        test_loader,
        id_to_token,
        SOS_ID,
        EOS_ID,
        PAD_ID,
        decode_fn
    )

    return cer

In [ ]:
configs = [
    {"use_ctc": True,  "use_coverage": True,  "decode_mode": "beam"},
    {"use_ctc": False, "use_coverage": True,  "decode_mode": "beam"},
    {"use_ctc": True,  "use_coverage": False, "decode_mode": "beam"},
    {"use_ctc": False, "use_coverage": False, "decode_mode": "beam"},
]

for cfg in configs:
    cer = evaluate_config(**cfg)
    print(cfg, "CER:", cer)


In [ ]:
def sample_qualitative_examples(
    model,
    loader,
    id_to_token,
    SOS_ID,
    EOS_ID,
    PAD_ID,
    decode_fn,
    num_examples=10
):
    model.eval()
    examples = []

    with torch.no_grad():
        for src, src_mask, tgt in loader:
            src = src.to(device)
            src_mask = src_mask.to(device)
            tgt = tgt.to(device)

            for i in range(src.size(0)):
                x = src[i:i+1]
                m = src_mask[i:i+1]
                y_true_ids = tgt[i].tolist()

                y_pred_ids = decode_fn(model, x, m)

                true_str = ids_to_string(y_true_ids, id_to_token, SOS_ID, EOS_ID, PAD_ID)
                pred_str = ids_to_string(y_pred_ids, id_to_token, SOS_ID, EOS_ID, PAD_ID)

                examples.append((true_str, pred_str))

                if len(examples) >= num_examples:
                    return examples

    return examples

# usage
examples = sample_qualitative_examples(
    ema_model,
    test_loader,
    id_to_token,
    SOS_ID,
    EOS_ID,
    PAD_ID,
    beam_decode
)

for i, (gt, pred) in enumerate(examples, 1):
    print(f"Example {i}")
    print("GT:   ", gt)
    print("Pred: ", pred)
    print("-" * 40)


In [ ]:
def plot_attention_heatmap(attn, src_mask, tgt_len, src_len):
    # attn: [layers, heads, tgt_len, src_len] or [heads, tgt_len, src_len]
    import seaborn as sns

    # example: use last layer, average heads
    if attn.dim() == 4:
        attn = attn[-1].mean(0)  # [tgt_len, src_len]

    attn = attn[:tgt_len, :src_len].cpu().numpy()

    plt.figure(figsize=(8, 6))
    sns.heatmap(attn, cmap="viridis")
    plt.xlabel("Source frames")
    plt.ylabel("Target positions")
    plt.title("Cross-attention")
    plt.show()


In [ ]:
def evaluate_with_perturbation(
    model,
    loader,
    id_to_token,
    SOS_ID,
    EOS_ID,
    PAD_ID,
    decode_fn,
    perturb_fn,
    label="perturbation"
):
    model.eval()
    total_dist = 0
    total_len = 0

    with torch.no_grad():
        for src, src_mask, tgt in loader:
            src = src.cpu().numpy()   # apply numpy-based perturbations
            src_mask = src_mask.numpy()
            tgt = tgt.to(device)

            for i in range(src.shape[0]):
                x_seq = src[i]  # [T, D]
                m_seq = src_mask[i]

                x_pert = perturb_fn(x_seq)  # must return [T', D]
                if x_pert is None:
                    continue

                T_new = x_pert.shape[0]
                x_tensor = torch.from_numpy(x_pert).unsqueeze(0).to(device)
                m_tensor = torch.ones(1, T_new, dtype=torch.bool, device=device)

                y_true_ids = tgt[i].tolist()
                y_pred_ids = decode_fn(model, x_tensor, m_tensor)

                true_str = ids_to_string(y_true_ids, id_to_token, SOS_ID, EOS_ID, PAD_ID)
                pred_str = ids_to_string(y_pred_ids, id_to_token, SOS_ID, EOS_ID, PAD_ID)

                if len(true_str) == 0:
                    continue

                dist = Lev.distance(pred_str, true_str)
                total_dist += dist
                total_len += len(true_str)

    cer = total_dist / total_len if total_len > 0 else 0.0
    print(f"[{label}] CER: {cer:.4f}")
    return cer


def perturb_subsample(seq):
    return seq[::2] if seq.shape[0] >= 20 else None

def perturb_dropout(seq):
    return motion_dropout(seq, drop_prob=0.1)

def perturb_warp(seq):
    return temporal_warp(seq)

evaluate_with_perturbation(
    ema_model,
    test_loader,
    id_to_token,
    SOS_ID,
    EOS_ID,
    PAD_ID,
    beam_decode,
    perturb_subsample,
    label="subsample x2"
)


In [ ]:
import time

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def benchmark_inference(
    model,
    loader,
    decode_fn,
    num_batches=10
):
    model.eval()
    start = time.time()
    n_samples = 0

    with torch.no_grad():
        for b, (src, src_mask, _) in enumerate(loader):
            if b >= num_batches:
                break
            src = src.to(device)
            src_mask = src_mask.to(device)

            for i in range(src.size(0)):
                x = src[i:i+1]
                m = src_mask[i:i+1]
                _ = decode_fn(model, x, m)
                n_samples += 1

    elapsed = time.time() - start
    if n_samples == 0:
        return None

    avg_time = elapsed / n_samples
    print(f"Inference: {avg_time:.4f} s/sample ({1.0/avg_time:.2f} samples/s)")
    return avg_time

# usage
n_params = count_parameters(model)
print("Trainable parameters:", n_params)

benchmark_inference(
    ema_model,
    test_loader,
    beam_decode,
    num_batches=5
)


In [ ]:
"""
Key observations:

- BLEU is computed using NLTK sentence_bleu
- CER uses Levenshtein distance normalized by length
- EXACT match is strict sequence equality
- Beam search improves accuracy but reduces diversity
- Encoder representation drift should stay stable
- Attention entropy can indicate overconfidence collapse

Potential improvements:

1. Relative positional encoding
2. Stronger data augmentation (temporal warp)
3. Label smoothing tuning
4. Better coverage penalty scheduling
5. Nucleus sampling decoding
6. Multi-signer conditioning (reduce bias)
"""